# 06 — Figure 2 at print resolution (arm examples)
**CPU is enough (about 2 min).** Input: the output of notebook 02 (`preprocess-cache-v2`), which contains `manifest_v2.csv` and `cache_*.npz`.
Same images as the QC figure of notebook 02 (3 pneumonia-positive images per cohort, `random_state=42`), redrawn at 600 dpi and
7.48 in wide (Elsevier full-page width; ≥ 2,244 px required for photographs).

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
INPUT, OUT, SEED = "/kaggle/input", "/kaggle/working", 42
plt.rcParams.update({"font.family": "DejaVu Sans", "pdf.fonttype": 42})
MAN = glob.glob(os.path.join(INPUT, "**", "manifest_v2.csv"), recursive=True)[0]; CDIR = os.path.dirname(MAN)
meta = pd.read_csv(MAN)
cohorts = [("internal", "Kermany internal"), ("rsna", "RSNA"), ("nih", "NIH ChestX-ray14"), ("vindr", "VinDr-CXR")]
store = {c: dict(np.load(os.path.join(CDIR, f"cache_{c}.npz"))) for c, _ in cohorts}
ARML = {"raw": "A raw", "roi": "B crop", "lung": "C mask"}

In [ ]:
fig, axs = plt.subplots(4, 9, figsize=(7.48, 3.75), gridspec_kw=dict(wspace=0.04, hspace=0.06))
for i, (c, name) in enumerate(cohorts):
    m = meta[(meta.cohort == c) & (meta.label == 1)]
    m = m[m.n_boxes > 0] if (m.n_boxes > 0).any() else m
    rows = m.sample(min(3, len(m)), random_state=SEED).row.tolist()
    for j in range(3):
        for k, a in enumerate(("raw", "roi", "lung")):
            ax = axs[i, j * 3 + k]; ax.set_axis_off(); rr = rows[j]
            ax.imshow(store[c][a][rr], cmap="gray", vmin=0, vmax=255, interpolation="lanczos")
            ax.contour(store[c]["m_" + a][rr], levels=[0.5], colors="#0E8FA0", linewidths=0.6)
            if store[c]["b_" + a][rr].any():
                ax.contour(store[c]["b_" + a][rr], levels=[0.5], colors="#F2C14E", linewidths=0.6, linestyles="--")
            if i == 0: ax.set_title(ARML[a], fontsize=7)
    axs[i, 0].text(-0.08, 0.5, name, transform=axs[i, 0].transAxes, rotation=90, ha="right", va="center", fontsize=7)
fig.subplots_adjust(left=0.035, right=0.995, top=0.94, bottom=0.005)
fig.savefig(os.path.join(OUT, "fig2_arms_examples.png"), dpi=600)
fig.savefig(os.path.join(OUT, "fig2_arms_examples.pdf"), dpi=600)
Image.open(os.path.join(OUT, "fig2_arms_examples.png")).convert("RGB").save(os.path.join(OUT, "fig2_arms_examples.tif"), compression="tiff_lzw", dpi=(600, 600))
print(Image.open(os.path.join(OUT, "fig2_arms_examples.png")).size, "pixels (must be ≥ 2244 wide)")